# Hybrid Text + Vision Retrieval — Best of Both Modalities

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/11_hybrid_text_plus_vision_retrieval.ipynb)

> **Orbit 5 (Multimodal)** · **Difficulty**: Intermediate · **Reading time**: ~30 min

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. **Explain** why single-modality retrieval (text-only or vision-only) has systematic blind spots on real-world documents.
2. **Design** a hybrid retrieval architecture that runs a text path and a vision path in parallel and merges their results.
3. **Implement** Reciprocal Rank Fusion (RRF) and linear score combination as fusion strategies.
4. **Build** an end-to-end `HybridRetriever` that combines sentence-transformer embeddings, VLM-generated captions, and FAISS indices.
5. **Evaluate** text-only, vision-only, and hybrid retrieval side-by-side to see when each strategy wins.

---
## Section 1 — Why Single-Modality Retrieval Has Blind Spots

Most retrieval systems bet on a single modality. A text-based pipeline extracts
words with OCR, chunks them, and embeds the chunks. A vision-based pipeline
embeds the raw page image or a VLM-generated caption. Each approach has real
strengths — and real weaknesses.

**Text retrieval misses visual content.** Charts, diagrams, photos, page layout,
colour-coding, and visual emphasis are invisible to a text-only index. A bar chart
showing quarterly revenue will be reduced to axis labels and numbers — losing the
trend, the comparison, and the visual salience that a human notices instantly.

**Vision retrieval misses precise textual detail.** VLM captions summarise what
the model *sees*, but they may drop specific numbers, names, dates, or acronyms.
A caption might say "a financial chart showing rising revenue" without mentioning
the exact figure of $42.7 M in Q3 2024.

**Real documents contain both modalities.** Slide decks pair bullet points with
charts. Technical reports embed diagrams next to equations. Medical records combine
text notes with scanned images. Relying on one modality means you miss retrieval
hits that only the *other* modality could have surfaced.

The hybrid approach we build in this notebook runs **both** retrieval paths in
parallel and fuses their results, so neither modality is a bottleneck.

### Illustrating the blind spots

We create a small synthetic document collection where some documents are
text-heavy, some are image-heavy, and some carry meaning in *both* modalities.
This lets us see exactly where each retrieval path fails.

In [ ]:
from PIL import Image, ImageDraw
import textwrap, os

# ── Synthetic document corpus ────────────────────────────────────────
DOC_TEXTS = [
    ("Total revenue reached $42.7M in Q3 2024, up 18% year-over-year. "
     "Cloud services contributed $28.1M while consulting brought $14.6M."),
    ("The system uses a three-tier architecture with load balancers, "
     "application servers, and a distributed database cluster."),
    ("Employees may work remotely up to 3 days per week. "
     "Manager approval is required for fully remote arrangements. "
     "All remote workers must use the company VPN."),
    ("Key milestones: v2.5 launch Oct 15, mobile app beta Nov 1, "
     "enterprise SSO integration Dec 10."),
    ("Hemoglobin: 13.2 g/dL (normal). White blood cells: 7,500/uL. "
     "Platelet count: 250,000/uL."),
    ("Primary colour: #0057B8. Secondary colour: #FFD700. "
     "Logo must have 20px minimum clear space."),
]

DOC_TITLES = [
    "Q3 2024 Revenue Report",
    "Network Architecture Diagram",
    "Employee Handbook - Remote Work Policy",
    "Product Roadmap Q4 2024",
    "Lab Results - Patient A",
    "Brand Guidelines",
]

DOC_VISUALS = ["bar_chart", "network_diagram", "plain_header",
               "timeline", "medical_chart", "colour_swatches"]

documents = []
for i, (title, text, vis) in enumerate(zip(DOC_TITLES, DOC_TEXTS, DOC_VISUALS)):
    documents.append({"id": f"doc_{i}", "title": title,
                      "text": text, "visual": vis})

print(f"Corpus size: {len(documents)} documents")
for d in documents:
    print(f"  {d['id']}: {d['title']}")

In [ ]:
def make_synthetic_image(doc, size=(400, 300)):
    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)
    vis = doc["visual"]

    if vis == "bar_chart":
        bars = [(60,180,110,280),(140,140,190,280),(220,100,270,280),(300,80,350,280)]
        for (x0,y0,x1,y1), c in zip(bars, ["#4e79a7","#59a14f","#f28e2b","#e15759"]):
            draw.rectangle([x0,y0,x1,y1], fill=c)
        draw.text((100,20), "Quarterly Revenue", fill="black")

    elif vis == "network_diagram":
        nodes = [(200,40,'LB'),(100,150,'App1'),(300,150,'App2'),(200,260,'DB')]
        for cx,cy,label in nodes:
            draw.rectangle([cx-30,cy-15,cx+30,cy+15], outline='black', width=2)
            draw.text((cx-10,cy-8), label, fill='black')
        for a,b in [((200,55),(100,135)),((200,55),(300,135)),
                     ((100,165),(200,245)),((300,165),(200,245))]:
            draw.line([a,b], fill='black', width=2)

    elif vis == "timeline":
        draw.line([(30,150),(370,150)], fill='black', width=3)
        for x,label in [(80,'Oct'),(200,'Nov'),(320,'Dec')]:
            draw.ellipse([x-6,144,x+6,156], fill='#e15759')
            draw.text((x-12,160), label, fill='black')

    elif vis == "colour_swatches":
        draw.rectangle([50,60,180,200], fill="#0057B8")
        draw.rectangle([220,60,350,200], fill="#FFD700")
        draw.text((80,220), "Primary", fill="black")
        draw.text((250,220), "Secondary", fill="black")

    elif vis == "medical_chart":
        import random; random.seed(42)
        pts = [(30+i*35, 150+random.randint(-40,40)) for i in range(10)]
        for i in range(len(pts)-1):
            draw.line([pts[i],pts[i+1]], fill='#4e79a7', width=2)
        draw.text((120,20), "Lab Results", fill="black")

    else:
        draw.rectangle([20,20,380,60], fill="#eeeeee")
        draw.text((30,30), doc['title'], fill='black')

    return img

images = {d['id']: make_synthetic_image(d) for d in documents}

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, doc in zip(axes, documents[:3]):
    ax.imshow(images[doc['id']])
    ax.set_title(doc['title'], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## Setup — Install Dependencies

In [ ]:
!pip install -q transformers torch qwen-vl-utils sentence-transformers faiss-cpu pillow bitsandbytes accelerate

---
## Section 2 — Hybrid Architecture Design

The core idea is simple: instead of choosing between text and vision, run **both**
retrieval paths for every query and merge the ranked lists at the end.

```
              ┌─────────────────────────┐
              │         Query            │
              └────────┬────────┬────────┘
                       │        │
           ┌───────────▼──┐  ┌──▼───────────┐
           │  Text Path   │  │ Vision Path   │
           │  (embed text) │  │ (embed caption│
           │  → FAISS      │  │  → FAISS)     │
           └───────┬──────┘  └──────┬────────┘
                   │                │
              ranked list A    ranked list B
                   │                │
              ┌────▼────────────────▼─────┐
              │     Score Fusion          │
              │  (Linear / RRF)           │
              └────────────┬──────────────┘
                           │
                   Final ranked list
```

**Text path.** Each document's raw text (or OCR output) is embedded with
`sentence-transformers/all-MiniLM-L6-v2` and stored in a FAISS `IndexFlatIP` index.

**Vision path.** Each document's image is captioned by `Qwen2.5-VL-7B-Instruct`,
and the resulting caption is embedded with the *same* sentence-transformer. This is the
**caption-bridge** approach from Notebook 09: convert vision to text, then embed.

**Fusion.** The two ranked lists are combined using either a linear weighted sum or
Reciprocal Rank Fusion (RRF). We implement both and compare.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')

bx = dict(boxstyle='round,pad=0.4', edgecolor='black', linewidth=1.5)

ax.text(5, 9.2, 'Query', ha='center', fontsize=12, fontweight='bold',
        bbox=dict(**bx, facecolor='#dbeafe'))
ax.text(2.5, 7, 'Text Path\n(embed text -> FAISS)', ha='center', fontsize=9,
        bbox=dict(**bx, facecolor='#dcfce7'))
ax.text(7.5, 7, 'Vision Path\n(caption -> embed -> FAISS)', ha='center', fontsize=9,
        bbox=dict(**bx, facecolor='#fef9c3'))
ax.text(5, 4, 'Score Fusion\n(Linear / RRF)', ha='center', fontsize=10,
        fontweight='bold', bbox=dict(**bx, facecolor='#fce7f3'))
ax.text(5, 1.8, 'Final Ranked Results', ha='center', fontsize=11,
        fontweight='bold', bbox=dict(**bx, facecolor='#e0e7ff'))

for s, e in [((5,8.8),(2.5,7.7)), ((5,8.8),(7.5,7.7)),
             ((2.5,6.3),(5,4.6)), ((7.5,6.3),(5,4.6)),
             ((5,3.4),(5,2.3))]:
    ax.annotate('', xy=e, xytext=s,
                arrowprops=dict(arrowstyle='->', color='grey', lw=1.5))

plt.title('Hybrid Retrieval Architecture', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

---
## Section 3 — Building the Text Retrieval Path

The text path is the simpler of the two. We take each document's raw text,
embed it with a sentence-transformer, and store the vectors in a FAISS inner-product
index. At query time we embed the query with the same model and retrieve the
top-*k* nearest neighbours.

**Why `all-MiniLM-L6-v2`?** It is small (22 M params), fast, and produces
384-dimensional embeddings that work well for semantic similarity. For a Colab demo
it strikes an excellent speed-quality balance.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f'Embedder loaded  dim={embedder.get_sentence_embedding_dimension()}')

In [ ]:
text_contents = [d['text'] for d in documents]
text_embeddings = embedder.encode(text_contents, normalize_embeddings=True)

dim = text_embeddings.shape[1]
text_index = faiss.IndexFlatIP(dim)
text_index.add(text_embeddings.astype(np.float32))

print(f'Text index: {text_index.ntotal} vectors, dim={dim}')

In [ ]:
def text_retrieve(query: str, top_k: int = 3):
    """Retrieve documents by text similarity."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = text_index.search(q_emb, top_k)
    return [
        {'doc_id': documents[i]['id'], 'title': documents[i]['title'],
         'score': float(s), 'source': 'text'}
        for s, i in zip(scores[0], indices[0])
    ]

# Quick test
for r in text_retrieve('What was Q3 revenue?'):
    print(f"  {r['doc_id']}  score={r['score']:.4f}  {r['title']}")

---
## Section 4 — Building the Vision Retrieval Path

For the vision path we use the **caption-bridge** strategy introduced in
Notebook 09. The idea: feed each document image through a VLM to produce a
detailed textual caption, then embed that caption with the same sentence-transformer.
This avoids needing a dedicated vision encoder for retrieval — we convert vision to
text and reuse the text-embedding infrastructure.

The VLM we use is `Qwen/Qwen2.5-VL-7B-Instruct`, loaded in 4-bit precision so it
fits on a T4 GPU. The generated captions capture visual layout, colours, chart
shapes, and diagram structure — exactly the information that OCR text misses.

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch
from qwen_vl_utils import process_vision_info

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map='auto',
    quantization_config=bnb_config, torch_dtype='auto',
)

print(f'VLM loaded: {MODEL_ID}')
print(f'GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

In [ ]:
def caption_image(image, prompt='Describe this image in detail.'):
    """Generate a caption for a PIL image using the VLM."""
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': prompt},
    ]}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors='pt',
    ).to(vlm.device)
    out = vlm.generate(**inputs, max_new_tokens=256)
    return processor.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

# Caption every document image
captions = {}
for doc in documents:
    cap = caption_image(images[doc['id']])
    captions[doc['id']] = cap
    print(f"{doc['id']}: {cap[:90]}...")

In [ ]:
caption_texts = [captions[d['id']] for d in documents]
caption_embeddings = embedder.encode(caption_texts, normalize_embeddings=True)

vision_index = faiss.IndexFlatIP(dim)
vision_index.add(caption_embeddings.astype(np.float32))
print(f'Vision index: {vision_index.ntotal} vectors')

In [ ]:
def vision_retrieve(query: str, top_k: int = 3):
    """Retrieve documents by vision-caption similarity."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = vision_index.search(q_emb, top_k)
    return [
        {'doc_id': documents[i]['id'], 'title': documents[i]['title'],
         'score': float(s), 'source': 'vision'}
        for s, i in zip(scores[0], indices[0])
    ]

# Quick test — a visually-oriented query
for r in vision_retrieve('chart showing bars going up'):
    print(f"  {r['doc_id']}  score={r['score']:.4f}  {r['title']}")

---
## Section 5 — Score Fusion Strategies

We now have two ranked lists for each query. How do we merge them into a single
result set? This is the **score fusion** problem, and it is more subtle than it looks.

### 5.1 — Linear Combination

The simplest approach: `final_score = α * text_score + (1 - α) * vision_score`.
The weight α controls the balance. Setting α = 0.5 gives equal weight; setting
α = 0.8 favours the text path. This is easy to understand and tune, but it has a
catch: the text and vision scores may live on **different scales**. FAISS inner-product
scores from two different indices are not directly comparable without normalisation.

### 5.2 — Reciprocal Rank Fusion (RRF)

RRF sidesteps the scale problem entirely. Instead of combining *scores*, it combines
*ranks*. For each document the RRF score is:

```
RRF(d) = Σ  1 / (k + rank_i(d))
```

where the sum runs over each retrieval list, `rank_i(d)` is the document's rank
(starting from 1), and `k` is a smoothing constant (typically 60). Documents that
appear near the top of *both* lists get the highest fused score.

**Why RRF is robust:** it requires no score normalisation across retrievers, it is
nearly parameter-free (only `k`), and empirical studies show it consistently
outperforms raw score combination across diverse retrieval settings.

In [ ]:
# ── 5.1  Linear combination fusion ──────────────────────────────────

def linear_fusion(text_results, vision_results, alpha=0.5):
    """Combine text and vision results via weighted score sum."""
    scores = {}
    for r in text_results:
        scores[r['doc_id']] = alpha * r['score']
    for r in vision_results:
        did = r['doc_id']
        scores[did] = scores.get(did, 0.0) + (1 - alpha) * r['score']
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [{'doc_id': d, 'score': s, 'source': 'hybrid-linear'} for d, s in ranked]

t_res = text_retrieve('quarterly revenue chart', top_k=4)
v_res = vision_retrieve('quarterly revenue chart', top_k=4)
fused = linear_fusion(t_res, v_res, alpha=0.5)

print('Linear fusion results:')
for r in fused[:4]:
    print(f"  {r['doc_id']}  score={r['score']:.4f}")

In [ ]:
# ── 5.2  Reciprocal Rank Fusion ─────────────────────────────────────

def reciprocal_rank_fusion(ranked_lists, k=60):
    """Merge multiple ranked lists using RRF."""
    scores = {}
    for ranked_list in ranked_lists:
        for rank, doc_id in enumerate(ranked_list):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

text_ids  = [r['doc_id'] for r in t_res]
vision_ids = [r['doc_id'] for r in v_res]

rrf_results = reciprocal_rank_fusion([text_ids, vision_ids], k=60)

print('RRF results:')
for doc_id, score in rrf_results[:4]:
    print(f'  {doc_id}  RRF_score={score:.6f}')

In [ ]:
# ── Compare fusion strategies on several queries ─────────────────────
queries = [
    'What was Q3 revenue?',
    'chart showing bars going up',
    'remote work policy details',
    'network architecture with load balancer',
    'brand primary colour blue swatch',
]

header = f'{"Query":<42} {"Linear #1":<16} {"RRF #1":<16}'
print(header)
print('-' * len(header))

for q in queries:
    t = text_retrieve(q, top_k=4)
    v = vision_retrieve(q, top_k=4)
    lin = linear_fusion(t, v, alpha=0.5)
    rrf = reciprocal_rank_fusion(
        [[r['doc_id'] for r in t], [r['doc_id'] for r in v]], k=60)
    print(f"{q:<42} {lin[0]['doc_id']:<16} {rrf[0][0]:<16}")

---
## Section 6 — End-to-End Hybrid Retrieval

Now we package everything into a single `HybridRetriever` class. The class
stores both FAISS indices, runs both retrieval paths on each query, applies the
chosen fusion strategy, and returns results annotated with their provenance —
so you can see whether a hit came from the text path, the vision path, or both.

In [ ]:
class HybridRetriever:
    """Parallel text + vision retrieval with score fusion."""

    def __init__(self, documents, text_index, vision_index, embedder,
                 fusion='rrf', alpha=0.5, rrf_k=60):
        self.documents = documents
        self.text_index = text_index
        self.vision_index = vision_index
        self.embedder = embedder
        self.fusion = fusion
        self.alpha = alpha
        self.rrf_k = rrf_k
        self._lookup = {d['id']: d for d in documents}

    def _search(self, index, query, top_k):
        q = self.embedder.encode([query], normalize_embeddings=True).astype(np.float32)
        scores, idxs = index.search(q, top_k)
        return [(self.documents[i]['id'], float(s))
                for s, i in zip(scores[0], idxs[0])]

    def retrieve(self, query, top_k=3):
        th = self._search(self.text_index, query, top_k + 2)
        vh = self._search(self.vision_index, query, top_k + 2)
        t_set = {d for d, _ in th}
        v_set = {d for d, _ in vh}

        if self.fusion == 'linear':
            sc = {}
            for d, s in th: sc[d] = self.alpha * s
            for d, s in vh: sc[d] = sc.get(d, 0.0) + (1 - self.alpha) * s
            ranked = sorted(sc.items(), key=lambda x: x[1], reverse=True)
        else:
            ranked = reciprocal_rank_fusion(
                [[d for d,_ in th], [d for d,_ in vh]], k=self.rrf_k)

        results = []
        for did, score in ranked[:top_k]:
            src = 'both' if did in t_set and did in v_set else (
                  'text-only' if did in t_set else 'vision-only')
            results.append({'doc_id': did,
                            'title': self._lookup[did]['title'],
                            'score': score, 'source': src})
        return results

In [ ]:
hybrid = HybridRetriever(
    documents, text_index, vision_index, embedder,
    fusion='rrf', rrf_k=60,
)

test_queries = [
    'What was total revenue in Q3 2024?',
    'diagram showing boxes connected by lines',
    'VPN requirement for remote workers',
    'timeline with milestones for product launch',
]

for q in test_queries:
    print(f'\nQuery: "{q}"')
    for r in hybrid.retrieve(q, top_k=3):
        print(f"  {r['doc_id']}  [{r['source']:<12}]  "
              f"score={r['score']:.6f}  {r['title']}")

---
## Section 7 — Evaluation: Text-Only vs Vision-Only vs Hybrid

Which approach wins? It depends on the query. Some queries are purely textual
("What is the VPN policy?"), some are purely visual ("which document has coloured
rectangles?"), and some require both modalities ("quarterly revenue bar chart
showing $42.7M"). We run all three retrieval modes and measure **Hit@1** — whether
the top-ranked result is the expected document.

This evaluation is small-scale but illustrative. In production you would use larger
benchmarks, nDCG, and recall@k. The key takeaway is consistent: hybrid retrieval
is the most *robust* across query types, because it never has a blind spot in an
entire modality.

In [ ]:
# ── Evaluation dataset + loop ────────────────────────────────────────
eval_queries = [
    # (query, expected_doc_id, dominant_modality)
    ('total revenue $42.7M Q3 2024',              'doc_0', 'text'),
    ('bar chart with four coloured bars',          'doc_0', 'vision'),
    ('three-tier architecture distributed database','doc_1', 'text'),
    ('boxes connected by lines diagram',           'doc_1', 'vision'),
    ('remote work VPN manager approval',           'doc_2', 'text'),
    ('timeline with red dots for months',          'doc_3', 'vision'),
    ('product roadmap v2.5 launch October',        'doc_3', 'text'),
    ('hemoglobin platelet count lab results',      'doc_4', 'text'),
    ('blue and gold colour swatches',              'doc_5', 'vision'),
    ('brand guidelines primary #0057B8',           'doc_5', 'text'),
]

def evaluate(retrieve_fn, queries):
    """Return Hit@1 for a retrieval function."""
    hits = 0
    for query, expected, _ in queries:
        res = retrieve_fn(query, top_k=1)
        top_id = res[0]['doc_id'] if isinstance(res[0], dict) else res[0][0]
        if top_id == expected:
            hits += 1
    return hits / len(queries)

text_hit1   = evaluate(text_retrieve, eval_queries)
vision_hit1 = evaluate(vision_retrieve, eval_queries)
hybrid_hit1 = evaluate(lambda q, k: hybrid.retrieve(q, k), eval_queries)

print(f'Hit@1  Text-only:     {text_hit1:.0%}')
print(f'Hit@1  Vision-only:   {vision_hit1:.0%}')
print(f'Hit@1  Hybrid (RRF):  {hybrid_hit1:.0%}')

In [ ]:
methods = ['Text-only', 'Vision-only', 'Hybrid (RRF)']
hit_scores = [text_hit1, vision_hit1, hybrid_hit1]
colours = ['#4e79a7', '#f28e2b', '#59a14f']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(methods, hit_scores, color=colours, edgecolor='black', linewidth=0.8)
ax.set_ylim(0, 1.1); ax.set_ylabel('Hit@1')
ax.set_title('Retrieval Accuracy by Strategy')
for bar, val in zip(bars, hit_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{val:.0%}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd

rows = []
for query, expected, modality in eval_queries:
    t = text_retrieve(query, top_k=1)
    v = vision_retrieve(query, top_k=1)
    h = hybrid.retrieve(query, top_k=1)
    rows.append({
        'query': query[:40], 'dominant': modality,
        'text_hit':   int(t[0]['doc_id'] == expected),
        'vision_hit': int(v[0]['doc_id'] == expected),
        'hybrid_hit': int(h[0]['doc_id'] == expected),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print('Aggregate by dominant modality:')
print(df.groupby('dominant')[['text_hit','vision_hit','hybrid_hit']].mean().round(2))

### When hybrid retrieval helps most

The pattern is clear: **text-only retrieval wins on text-dominant queries** and
**vision-only retrieval wins on visual queries**, but **hybrid wins (or ties) on both**.
In real-world document collections — slide decks, financial reports, technical
manuals, medical records — queries are a mix of both modalities, so hybrid is the
safest default strategy.

---
## Section 8 — Connections to the Castalia Curriculum

The score-fusion techniques here — linear combination and RRF — are the same ones
used in the RAG orbit:

- **`rag/fusion_retrieval.ipynb`** applies RRF to merge dense and sparse (BM25)
  retrieval over pure text.
- **`rag/ensemble_retrieval.ipynb`** explores ensemble strategies with multiple
  embedding models.

This notebook extends those ideas to the **multimodal** setting: instead of fusing
*dense vs sparse*, we fuse *text vs vision*. The same `reciprocal_rank_fusion`
function works unchanged — a testament to how composable the RRF abstraction is.

In production systems you might add more retrieval paths (e.g., keyword search,
knowledge-graph lookup) and fuse them all with RRF. The pattern scales.

---
## 🏋️ Exercises

### Exercise 1 — Tune the fusion weight α

Sweep α from 0.0 to 1.0 and plot Hit@1 vs α. At what value does linear fusion
peak? How does it compare to RRF (which is parameter-free)?

In [ ]:
# Exercise 1 — Starter code
alphas = np.arange(0.0, 1.05, 0.1)
linear_scores = []

for a in alphas:
    h = HybridRetriever(
        documents, text_index, vision_index, embedder,
        fusion='linear', alpha=a,
    )
    score = evaluate(lambda q, k: h.retrieve(q, k), eval_queries)
    linear_scores.append(score)

# TODO: plot alphas vs linear_scores
# TODO: add a horizontal line for the RRF Hit@1 score

### Exercise 2 — Add a keyword (BM25) retrieval path

Install `rank_bm25`, build a BM25 index over the document texts, and add it as a
third path in the RRF fusion. Does three-way fusion beat two-way?

In [ ]:
# Exercise 2 — Starter code
# !pip install -q rank_bm25
# from rank_bm25 import BM25Okapi
#
# tokenized = [doc['text'].lower().split() for doc in documents]
# bm25 = BM25Okapi(tokenized)
#
# def bm25_retrieve(query, top_k=3):
#     tok_q = query.lower().split()
#     scores = bm25.get_scores(tok_q)
#     top_idx = scores.argsort()[::-1][:top_k]
#     return [documents[i]['id'] for i in top_idx]
#
# # Fuse three lists with RRF:
# # reciprocal_rank_fusion([text_ids, vision_ids, bm25_ids])

### Exercise 3 — Vision-aware re-ranking

After fusion, take the top-5 results and re-rank them by asking the VLM:
"Does this image answer the query: {query}? Rate relevance 1-5."
Parse the score and re-sort. Does VLM re-ranking improve Hit@1?

In [ ]:
# Exercise 3 — Starter code
# def vlm_rerank(query, candidate_ids, top_k=3):
#     scored = []
#     for did in candidate_ids:
#         prompt = (
#             f'Rate how relevant this image is to: "{query}". '
#             'Reply with a single integer from 1 to 5.'
#         )
#         text = caption_image(images[did], prompt=prompt)
#         try:
#             digits = ''.join(c for c in text if c.isdigit())
#             rating = int(digits[:1]) if digits else 1
#         except ValueError:
#             rating = 1
#         scored.append((did, rating))
#     scored.sort(key=lambda x: x[1], reverse=True)
#     return [did for did, _ in scored[:top_k]]

---
## 📋 Key Takeaways

| # | Insight |
|---|---------|
| 1 | **Text retrieval has blind spots** — it cannot see charts, diagrams, layout, or colour. |
| 2 | **Vision retrieval has blind spots** — VLM captions may drop specific numbers, names, and dates. |
| 3 | **Hybrid retrieval** runs both paths in parallel and fuses the results, covering both blind spots. |
| 4 | **RRF** is the safest fusion strategy because it works on ranks, not raw scores, and needs no normalisation. |
| 5 | **Linear fusion** is simpler but requires tuning α and normalising scores to a common scale. |
| 6 | The caption-bridge approach (Notebook 09) lets you reuse text-embedding infrastructure for vision retrieval — no separate image encoder needed. |

---
## 📚 References

1. Cormack, G. V., Clarke, C. L. A., & Buttcher, S. (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods.* SIGIR 2009.
2. Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* EMNLP 2019.
3. Qwen Team (2025). *Qwen2.5-VL Technical Report.* arXiv:2502.13923.
4. Johnson, J., Douze, M., & Jegou, H. (2019). *Billion-scale similarity search with GPUs.* IEEE Transactions on Big Data.
5. Castalia RAG Orbit — `rag/fusion_retrieval.ipynb`, `rag/ensemble_retrieval.ipynb`.

---

< **Previous**: [10 — Page-as-Image Retrieval with ColPali](10_page_as_image_retrieval_with_colpali.ipynb)

> **Next**: [12 — Multimodal Grounding and Evaluation](12_multimodal_grounding_and_evaluation.ipynb)